# Discrete Kantorovich dual potentials

This notebook generates `fig:dual-kantorovich-discrete-potentials`.  For a discrete quadratic transport problem
$$
    \min_{P\in U(a,b)} \langle C,P\rangle,
    \qquad C_{ij}=|x_i-y_j|^2,
$$
the dual variables satisfy
$$
    f_i+g_j\leq C_{ij},
    \qquad \max_{f,g}\; \langle f,a\rangle+\langle g,b\rangle .
$$
The panels display a fixed central source histogram and three two-mode targets.  The lower curves are optimal dual potentials in a common gauge.

In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import linprog

from figure_style import (
    RED, BLUE, VIOLET, GRAY, LIGHT_GRAY,
    DIRAC_MARKER_SIZE, setup_matplotlib, figure_dir, save_pdf, box_axes,
)

setup_matplotlib()

## Histograms and finite dual linear program

The problem size is intentionally small: about thirty bins are enough for a readable LP certificate.  We solve the dual linear program directly, then apply one pair of hard `c`-transforms so that the displayed potentials are tight best responses.

In [2]:
NAME = "dual-kantorovich-discrete-potentials"
OUT = figure_dir(NAME)

grid = np.linspace(-2.55, 2.55, 31)
dx = grid[1] - grid[0]


def gaussian_mixture_density(x, weights, means, stds):
    x = np.asarray(x)
    density = np.zeros_like(x, dtype=float)
    for w, m, s in zip(weights, means, stds):
        density += w * np.exp(-0.5 * ((x - m) / s) ** 2) / (s * np.sqrt(2 * np.pi))
    return density


def histogram(weights, means, stds):
    density = gaussian_mixture_density(grid, weights, means, stds)
    mass = density / density.sum()
    return density, mass


def solve_discrete_dual(a, b, C):
    n, m = C.shape
    objective = -np.r_[a, b]
    A = np.zeros((n * m, n + m))
    row = 0
    for i in range(n):
        for j in range(m):
            A[row, i] = 1.0
            A[row, n + j] = 1.0
            row += 1
    bounds = [(None, None)] * (n + m)
    bounds[0] = (0.0, 0.0)  # fix the additive gauge during the LP solve
    result = linprog(objective, A_ub=A, b_ub=C.ravel(), bounds=bounds, method="highs")
    if not result.success:
        raise RuntimeError(result.message)
    f = result.x[:n]
    g = result.x[n:]
    # Replace by tight best responses; the gauge is reset afterwards.
    g = np.min(C - f[:, None], axis=0)
    f = np.min(C - g[None, :], axis=1)
    shift = np.sum(a * f)
    f = f - shift
    g = g + shift
    return f, g

source_params = dict(weights=[1.0], means=[0.0], stds=[0.56])
targets = [
    ("balanced", dict(weights=[0.50, 0.50], means=[-1.15, 1.12], stds=[0.24, 0.24])),
    ("shifted", dict(weights=[0.34, 0.66], means=[-1.42, 0.58], stds=[0.30, 0.25])),
    ("unequal", dict(weights=[0.58, 0.42], means=[-0.76, 1.55], stds=[0.48, 0.18])),
]

alpha_density, a = histogram(**source_params)
C = (grid[:, None] - grid[None, :]) ** 2
solutions = []
for tag, params in targets:
    beta_density, b = histogram(**params)
    f, g = solve_discrete_dual(a, b, C)
    solutions.append((tag, beta_density, b, f, g))

all_potentials = np.concatenate([np.r_[f, g] for _, _, _, f, g in solutions])
ymin, ymax = np.percentile(all_potentials, [1, 99])
margin = 0.12 * (ymax - ymin)
ylim_potential = (ymin - margin, ymax + margin)
max_mass = max(alpha_density.max(), *(beta_density.max() for _, beta_density, _, _, _ in solutions))

## Exported panels

Each PDF contains only the histograms and potential curves.  The panel labels and the explanation of the gauge, colors, and cost are supplied by the LaTeX figure caption.

In [3]:
def draw_panel(tag, beta_density, f, g):
    fig = plt.figure(figsize=(2.36, 1.96))
    gs = fig.add_gridspec(2, 1, height_ratios=[0.62, 1.0], hspace=0.08)
    ax_m = fig.add_subplot(gs[0, 0])
    ax_p = fig.add_subplot(gs[1, 0], sharex=ax_m)

    ax_m.bar(grid, alpha_density, width=0.82 * dx, color=RED, alpha=0.34, linewidth=0)
    ax_m.bar(grid, -beta_density, width=0.82 * dx, color=BLUE, alpha=0.34, linewidth=0)
    ax_m.axhline(0, color="#777777", lw=0.45)
    ax_m.set_xlim(grid.min(), grid.max())
    ax_m.set_ylim(-1.08 * max_mass, 1.08 * max_mass)
    ax_m.tick_params(labelbottom=False, labelleft=False, length=0)
    box_axes(ax_m)

    ax_p.plot(grid, f, color=RED, lw=1.12, marker="o", markersize=2.2, markeredgewidth=0)
    ax_p.plot(grid, g, color=BLUE, lw=1.12, marker="o", markersize=2.2, markeredgewidth=0)
    ax_p.axhline(0, color=LIGHT_GRAY, lw=0.55, zorder=0)
    ax_p.set_xlim(grid.min(), grid.max())
    ax_p.set_ylim(*ylim_potential)
    ax_p.tick_params(labelbottom=False, labelleft=False, length=0)
    box_axes(ax_p)

    save_pdf(fig, OUT / f"target-{tag}.pdf", pad_inches=0.035)
    plt.close(fig)

for tag, beta_density, b, f, g in solutions:
    draw_panel(tag, beta_density, f, g)